# 04 — Aprendizaje supervisado (ocurrencia de incendios)

## [ESP] Objetivo

Ranking de riesgo de ocurrencia a partir de **condiciones cotidianas**
(no satelitales). Positivos = detecciones FIRMS
(`data/processed/expanded/firms_spain_final_lags_index.csv`); negativos =
días de fondo enriquecidos (`background_days.csv`, clave `background_id`).
Métrica principal **PR-AUC**; secundaria ROC-AUC. El modelo produce un
**ranking**, no una probabilidad: la prevalencia la fija el muestreo de
negativos, no la naturaleza (ver `planning/plan_supervised_implementation.md`,
Plan D). Como anexo **provisional**, la sección 8 calibra una probabilidad de
**excedencia a margen ancho** (P(≥1 detección a ≤ R km ese día), R∈{5,10,20} km).

## [ENG] Objective

Risk ranking of fire occurrence from everyday (non-satellite) conditions.
Positives = FIRMS detections; negatives = enriched background days. Primary
metric PR-AUC; no calibrated probabilities.

## Reglas (recordatorio)

- Validación **bloqueada** (leave-one-group-out) por temporada y CCAA: nada de
  K-fold aleatorio (fuga entre días/lugares cercanos).
- Semilla fija `SEED = 42`; unidades crudas; **ninguna columna de satélite**
  entra en `X` (`occurrence_features` las descarta con aviso).
- El one-hot y el escalador se ajustan **solo en el pliegue de entrenamiento**
  y se alinean entre pliegues (categorías no vistas → todo-cero).
- Dependencia: el fondo enriquecido lo construye el equipo. Mientras no exista,
  esta celda 1 fallará en `load_background_days()`.

In [1]:
# Imports del cuaderno (celda unica). Todas las dependencias aqui; las demas
# celdas no importan nada.
from IPython.display import display
import matplotlib
import matplotlib.pyplot as plt
import geopandas
import sklearn
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.inspection import permutation_importance
from sklearn.metrics import PrecisionRecallDisplay
from wildfire.analysis.ablation import ablation_verdict
from wildfire.analysis.calibration import (
    calibrate_exceedance,
    dedupe_events,
    exceedance_summary,
)
from wildfire.analysis.deployment import (
    add_risk_bands,
    ccaa_day_fire_rates,
    join_ccaa_fire_rates,
)
from wildfire.analysis.occurrence_model import (
    categorical_columns,
    categorical_to_dummies,
    continuous_columns,
    fold_scaler_apply,
    fold_scaler_fit,
    model_candidates,
)
from wildfire.data.io import load_csv, save_csv
from wildfire.geo.regions import REGION_NAME_MAP, load_boundaries
from wildfire.processing.occurrence import (
    BACKGROUND_ID_COL,
    EVERYDAY_FEATURES,
    LABEL_COL,
    SATELLITE_BLOCKLIST,
    SEASON_COL,
    SUBDAILY_DROP,
    add_season,
    assemble_modeling_frame,
    blocked_splits,
    load_background_days,
    occurrence_features,
    occurrence_settings,
    rank_metrics,
    sample_background,
    validate_background_days,
)
from wildfire.processing.reintegration import fit_background_scorer

# Rutas (relativas a PROJECT_ROOT; load_csv/save_csv las anclan) y semilla.
POSITIVES_CSV = "data/processed/expanded/firms_spain_final_lags_index.csv"
PREDICTIONS_CSV = "data/processed/predictions/occurrence_scores.csv"
FOLD_METRICS_CSV = "data/processed/predictions/occurrence_fold_metrics.csv"
CCAA_RATES_CSV = "data/processed/predictions/ccaa_day_fire_rates.csv"
CALIB_CSV = "data/processed/predictions/calibration_reliability.csv"
EVENTOS_CSV = "data/processed/unsupervised/eventos.csv"
SEED = 42
CANDIDATES = ["cluster_meteo", "score_anomalia"]

settings = occurrence_settings()
RATIO = settings["background_ratio"]
LEAKAGE_KM = settings["leakage_max_km"]
RISK_BANDS = settings["risk_bands"]
MARGINS = tuple(settings["calibration_margins_km"])
PROTOCOLS = tuple(settings["cv_groups"])
print(f"ratio={RATIO} | leakage_max_km={LEAKAGE_KM} | protocolos={PROTOCOLS} | seed={SEED}")



ratio=2 | leakage_max_km=1.0 | protocolos=('season', 'ccaa') | seed=42


## 1 — Carga, validación y ensamblado

Carga positivos y fondo enriquecido, valida el contrato del fondo (clave
única, esquema cotidiano, sin satélite, sin fugas a ≤ `leakage_max_km` de una
detección del mismo día), muestrea negativos a `ratio` por positivo y ensambla
la matriz de modelado. `season` y `record_id` se re-adjuntan por posición
(`assemble_modeling_frame` solo conserva `EVERYDAY_FEATURES` + `label`).

_[ENG] Load positives + enriched background; validate the background contract (unique key, everyday schema, no satellite, no same-day leakage); sample negatives at the configured ratio and assemble the modeling frame._

In [2]:
positives = load_csv(POSITIVES_CSV)
if "detection_id" not in positives.columns:
    raise KeyError("Los positivos necesitan la columna detection_id")

background = load_background_days()
report = validate_background_days(background, positives, leakage_max_km=LEAKAGE_KM)
print(
    f"Fondo validado: {report['n_rows']:,} filas | "
    f"duplicados={report['n_duplicate_ids']} | fugas={report['n_leakage']}"
)
if report["nan_report"]:
    display(pd.Series(report["nan_report"], name="NaN").rename_axis("columna").to_frame())
else:
    print("Sin NaN en el fondo.")

sample = sample_background(background, n_positives=len(positives), ratio=RATIO, seed=SEED)
requested = RATIO * len(positives)
note = "" if len(sample) == requested else " (pool insuficiente: se usa todo)"
print(
    f"Positivos: {len(positives):,} | fondo: {len(background):,} | "
    f"pedido ratio {RATIO} = {requested:,} | muestreado: {len(sample):,}{note}"
)

pos_season = add_season(positives)[SEASON_COL].to_numpy()
neg_season = add_season(sample)[SEASON_COL].to_numpy()

# occurrence_features descarta columnas de satelite/sub-diarias con aviso.
pos_features = occurrence_features(positives)
neg_features = occurrence_features(sample)
frame = assemble_modeling_frame(pos_features, neg_features)
frame[SEASON_COL] = np.concatenate([pos_season, neg_season])
frame["record_id"] = np.concatenate(
    [
        positives["detection_id"].astype(str).to_numpy(),
        sample[BACKGROUND_ID_COL].astype(str).to_numpy(),
    ]
)

print("Balance de etiquetas:")
display(frame[LABEL_COL].value_counts().rename("n").rename_axis("label").to_frame())
print("Etiquetas por CCAA:")
display(frame.groupby("ccaa")[LABEL_COL].value_counts().unstack(fill_value=0))
print("Etiquetas por temporada:")
display(frame.groupby(SEASON_COL)[LABEL_COL].value_counts().unstack(fill_value=0))

Fondo validado: 23,895 filas | duplicados=0 | fugas=0
Sin NaN en el fondo.
Positivos: 23,142 | fondo: 23,895 | pedido ratio 2 = 46,284 | muestreado: 23,895 (pool insuficiente: se usa todo)
Balance de etiquetas:


C:\Projects\wildfires-project\src\wildfire\processing\occurrence.py:226: UserWarning: Ignoring 13 satellite/sub-daily columns (['acq_time', 'brightness', 'brightness_ir', 'confidence_cat', 'confidence_num', 'daynight', 'franja_horaria', 'frp', 'is_day', 'rain', 'scan', 'type']); they cannot enter the occurrence model
  warnings.warn(
C:\Projects\wildfires-project\src\wildfire\processing\occurrence.py:226: UserWarning: Ignoring 7 satellite/sub-daily columns (['acq_time', 'daynight', 'franja_horaria', 'is_day', 'rain', 'type']); they cannot enter the occurrence model
  warnings.warn(


,n
label,
0,23895
1,23142


Etiquetas por CCAA:


label,0,1
ccaa,,
Andalucía,3995,3299
Aragón,2348,714
Asturias,536,4266
Canarias,328,2733
Cantabria,286,1029
Castilla y León,4530,1744
Castilla-La Mancha,3675,1583
Cataluña,1515,731
Comunidad Valenciana,1075,1539


Etiquetas por temporada:


label,0,1
season,,
2023-Q1,2932,5636
2023-Q2,3029,3146
2023-Q3,3028,4638
2023-Q4,2991,1274
2024-Q1,3044,1114
2024-Q2,2966,1947
2024-Q3,2960,4438
2024-Q4,2945,949


## 2 — Línea base bloqueada

`fold_eval` ajusta, por pliegue (leave-one-group-out sobre season/ccaa), el
one-hot y el escalador **solo con entrenamiento**, entrena el modelo y calcula
PR-AUC/ROC-AUC en el grupo retenido. La **PR-AUC de línea base = prevalencia**
del pliegue (un clasificador sin habilidad obtiene exactamente eso). Se
descartan pliegues de una sola clase.

_[ENG] Blocked leave-one-group-out baseline: per-fold one-hot/scaler fit on the train fold only; the PR-AUC baseline equals fold prevalence; single-class folds are skipped._

In [3]:
def prepare_fold(train, test, feature_cols, categorical_cols):
    """One-hot (ajustado en train) + imputacion de medianas + escalado por pliegue."""
    X_train, dummies = categorical_to_dummies(train[feature_cols], categorical_cols)
    X_test, _ = categorical_to_dummies(
        test[feature_cols], categorical_cols, train_columns=dummies
    )
    cont = continuous_columns(X_train)
    medians = X_train[cont].median()
    X_train[cont] = X_train[cont].fillna(medians)
    X_test[cont] = X_test[cont].fillna(medians)
    scaler = fold_scaler_fit(X_train, cont)
    return fold_scaler_apply(X_train, scaler, cont), fold_scaler_apply(X_test, scaler, cont)


def fold_eval(data, feature_cols, categorical_cols, model, protocols=PROTOCOLS):
    """PR-AUC/ROC-AUC por pliegue para un modelo (leave-one-group-out)."""
    rows = []
    for protocol in protocols:
        for split in blocked_splits(data, protocol):
            train, test = data.iloc[split.train_index], data.iloc[split.test_index]
            if train[LABEL_COL].nunique() < 2 or test[LABEL_COL].nunique() < 2:
                print(f"[skip] {protocol}/{split.held_out}: menos de dos clases")
                continue
            X_train, X_test = prepare_fold(train, test, feature_cols, categorical_cols)
            fitted = clone(model).fit(X_train, train[LABEL_COL])
            scores = fitted.predict_proba(X_test)[:, 1]
            rows.append(
                {
                    "protocol": protocol,
                    "held_out": split.held_out,
                    "n_train": len(train),
                    "n_test": len(test),
                    "prevalence": float(test[LABEL_COL].mean()),
                    **rank_metrics(test[LABEL_COL], scores),
                }
            )
    return pd.DataFrame(rows)


floor_name = "logistic"
floor_model = model_candidates(seed=SEED)[floor_name]
baseline_metrics = fold_eval(frame, EVERYDAY_FEATURES, categorical_columns(), floor_model)
display(baseline_metrics)
display(baseline_metrics.groupby("protocol")[["pr_auc", "roc_auc"]].agg(["mean", "min"]))
print(f"PR-AUC sin habilidad (prevalencia media): {baseline_metrics['prevalence'].mean():.3f}")

,protocol,held_out,n_train,n_test,prevalence,pr_auc,roc_auc
0,season,2023-Q1,38469,8568,0.657796,0.942580,0.909580
1,season,2023-Q2,40862,6175,0.509474,0.840258,0.855671
2,season,2023-Q3,39371,7666,0.605009,0.661114,0.569081
3,season,2023-Q4,42772,4265,0.298710,0.725159,0.881499
4,season,2024-Q1,42879,4158,0.267917,0.665769,0.851051
5,season,2024-Q2,42124,4913,0.396296,0.701903,0.778617
6,season,2024-Q3,39639,7398,0.599892,0.703294,0.669583
7,season,2024-Q4,43143,3894,0.243708,0.541627,0.835980
8,ccaa,Andalucía,39743,7294,0.452290,0.677537,0.758576
9,ccaa,Aragón,43975,3062,0.233181,0.431196,0.753467


pr_auc             roc_auc          
              mean       min      mean       min
protocol                                        
ccaa      0.670579  0.366766  0.768972  0.616678
season    0.722713  0.541627  0.793883  0.569081

PR-AUC sin habilidad (prevalencia media): 0.456


## 3 — Comparación de modelos candidatos

Los tres modelos provisionales (`LogisticRegression`, `RandomForestClassifier`,
`HistGradientBoostingClassifier`) bajo los mismos pliegues. Media/mínimo de
PR-AUC y ROC-AUC por modelo y protocolo. El mejor por PR-AUC media se usa en
la ablación.

_[ENG] The three provisional models under identical folds; compare mean/min PR-AUC and ROC-AUC; the best by mean PR-AUC feeds the ablation._

In [4]:
comparisons = []
for name, model in model_candidates(seed=SEED).items():
    table = fold_eval(frame, EVERYDAY_FEATURES, categorical_columns(), model)
    table["model"] = name
    comparisons.append(table)
candidate_metrics = pd.concat(comparisons, ignore_index=True)
comparison_summary = (
    candidate_metrics.groupby("model")[["pr_auc", "roc_auc"]]
    .agg(["mean", "min"])
    .sort_values(("pr_auc", "mean"), ascending=False)
)
display(comparison_summary)
best_name = comparison_summary.index[0]
print(f"Mejor modelo por PR-AUC media: {best_name}")

pr_auc             roc_auc          
                            mean       min      mean       min
model                                                         
hist_gradient_boosting  0.850071  0.666817  0.887006  0.700340
random_forest           0.809842  0.630062  0.855865  0.565187
random_forest_balanced  0.807956  0.636541  0.854612  0.529415
logistic_balanced       0.687389  0.367324  0.776975  0.568544
logistic                0.687262  0.366766  0.776943  0.569081

Mejor modelo por PR-AUC media: hist_gradient_boosting


## 4 — Reintegración (block 03 → 04)

`fit_background_scorer(positives)` ajusta **solo con positivos** el PCA
meteorológico, la tipología k12 y los detectores de anomalías; `.score` etiqueta
(`cluster_meteo`) y puntúa (`score_anomalia`) ambas clases con esos mismos
objetos (nunca reajusta). Se reconstruye la matriz con los candidatos
(`pos_candidates`/`neg_candidates`).

_[ENG] Fit the background scorer on positives only and score both classes; rebuild the frame with the cluster_meteo/score_anomalia candidates._

In [5]:
scorer = fit_background_scorer(positives, seed=SEED)
pos_candidates = scorer.score(positives)[CANDIDATES]
neg_candidates = scorer.score(sample)[CANDIDATES]

frame_candidates = assemble_modeling_frame(
    pos_features,
    neg_features,
    pos_candidates=pos_candidates,
    neg_candidates=neg_candidates,
)
frame_candidates[SEASON_COL] = frame[SEASON_COL].to_numpy()
frame_candidates["record_id"] = frame["record_id"].to_numpy()

display(frame_candidates[CANDIDATES].describe())
print("Medias por clase:")
display(frame_candidates.groupby(LABEL_COL)[CANDIDATES].mean())

,cluster_meteo,score_anomalia
count,47037.000000,47037.000000
mean,6.155962,0.068769
std,3.191555,0.079477
min,0.000000,0.000000
25%,3.000000,0.033853
50%,6.000000,0.050248
75%,9.000000,0.074192
max,11.000000,1.000000


Medias por clase:


,cluster_meteo,score_anomalia
label,,
0,6.405148,0.082154
1,5.898669,0.054948


## 5 — Ablación (puerta de promoción)

Con el mejor modelo base, bajo ambos protocolos: con vs sin
`cluster_meteo` + `score_anomalia`. `ablation_verdict` calcula ΔPR-AUC por
pliegue, media, **IC bootstrap** (semilla 42) y mínimo por protocolo. Se
promueve solo si la media supera el margen, el IC excluye 0 y ningún protocolo
queda negativo. Controles negativos: se añade `month`/`ccaa` one-hot como si
fuesen candidatos (no deben promoverse).

_[ENG] Ablation gate for the candidates under both protocols (mean delta PR-AUC, bootstrap CI, per-protocol min); month/ccaa one-hot additions act as negative controls._

In [6]:
candidate_feature_cols = EVERYDAY_FEATURES + [c for c in CANDIDATES if c in frame_candidates.columns]
candidate_cat_cols = [
    c for c in categorical_columns(include_candidates=True) if c in candidate_feature_cols
]
best_model = model_candidates(seed=SEED)[best_name]

with_metrics = fold_eval(frame_candidates, candidate_feature_cols, candidate_cat_cols, best_model)
without_metrics = fold_eval(frame_candidates, EVERYDAY_FEATURES, categorical_columns(), best_model)
verdict = ablation_verdict(with_metrics, without_metrics, seed=SEED)
display(pd.DataFrame([verdict]))

# Controles negativos por el mismo veredicto.
frame_controls = frame_candidates.copy()
frame_controls["control_month"] = frame_controls["month"].astype(str)
frame_controls["control_ccaa"] = frame_controls["ccaa"].astype(str)
for control in ("control_month", "control_ccaa"):
    ctrl_with = fold_eval(
        frame_controls,
        EVERYDAY_FEATURES + [control],
        categorical_columns() + [control],
        best_model,
    )
    ctrl_verdict = ablation_verdict(ctrl_with, without_metrics, seed=SEED)
    print(
        f"Control {control}: promovido={ctrl_verdict['promoted']} "
        f"(delta medio {ctrl_verdict['mean_delta']:+.4f})"
    )

print(
    f"Candidatos promovidos: {verdict['promoted']} "
    f"(delta medio {verdict['mean_delta']:+.4f}, "
    f"IC95% [{verdict['ci_low']:+.4f}, {verdict['ci_high']:+.4f}])"
)

,mean_delta,ci_low,ci_high,per_protocol_mean,per_protocol_min,per_fold_min,n_folds,margin,promoted
0,0.003901,-0.003275,0.011596,"{'ccaa': 0.00506510835564827, 'season': 0.0014...","{'ccaa': -0.02485772492304561, 'season': -0.03...",-0.038968,25,0.0,False


Control control_month: promovido=False (delta medio -0.0009)


Control control_ccaa: promovido=False (delta medio +0.0000)
Candidatos promovidos: False (delta medio +0.0039, IC95% [-0.0033, +0.0116])


## 6 — Auditoría del pliegue anómalo (2024-Q1)

El pliegue `season/2024-Q1` puntúa casi perfecto (PR-AUC ~0.99) frente a
`2023-Q1`. Antes de sacar conclusiones se auditan cuatro cosas: que la matriz
`X` no traiga columnas de satélite/sub-diarias (prohibidas en el contrato), que
un control trivial (mes/día-del-año/CCAA) no reproduzca el score, el perfil de
features por clase y la importancia por permutación en ese pliegue.

_[ENG] Audit the anomalously good season/2024-Q1 fold: banned columns in X, a trivial calendar/CCAA control, a per-class feature profile and the fold's permutation importance._

In [7]:
# Auditoría del pliegue anómalo (season/2024-Q1). El pliegue sale casi
# perfecto (PR-AUC ~0.99) frente a 2023-Q1: comprobamos que NO es fuga de
# etiqueta ni un artefacto de calendario/CCAA.
AUDIT_FOLD = "2024-Q1"
PEER_FOLD = "2023-Q1"
CONTROL_COLS = ["month", "day_of_year", "ccaa"]
CONTROL_CAT = ["ccaa"]


def eval_one_fold(data, protocol, held_out, feature_cols, cat_cols, model):
    """Modelo ajustado + train/test + X_test + scores de un pliegue bloqueado."""
    split = next(s for s in blocked_splits(data, protocol) if s.held_out == held_out)
    train, test = data.iloc[split.train_index], data.iloc[split.test_index]
    X_train, X_test = prepare_fold(train, test, feature_cols, cat_cols)
    fitted = clone(model).fit(X_train, train[LABEL_COL])
    return fitted, train, test, X_test, fitted.predict_proba(X_test)[:, 1]


audit_model = model_candidates(seed=SEED)[best_name]

audit_rows = []
for held_out in (AUDIT_FOLD, PEER_FOLD):
    _, _, test, X_test, scores = eval_one_fold(
        frame, "season", held_out, EVERYDAY_FEATURES, categorical_columns(), audit_model
    )
    _, _, test_c, _, scores_c = eval_one_fold(
        frame, "season", held_out, CONTROL_COLS, CONTROL_CAT, audit_model
    )
    audit_rows.append(
        {
            "held_out": held_out,
            "n_test": len(test),
            "prevalence": float(test[LABEL_COL].mean()),
            **rank_metrics(test[LABEL_COL], scores),
            "control_pr_auc": rank_metrics(test_c[LABEL_COL], scores_c)["pr_auc"],
            "control_roc_auc": rank_metrics(test_c[LABEL_COL], scores_c)["roc_auc"],
        }
    )
audit_table = pd.DataFrame(audit_rows)
display(audit_table)

fitted_q1, train_q1, test_q1, X_q1, scores_q1 = eval_one_fold(
    frame, "season", AUDIT_FOLD, EVERYDAY_FEATURES, categorical_columns(), audit_model
)

# (1) Fuga: X no puede contener columnas de satélite/sub-diarias.
banned = [c for c in SATELLITE_BLOCKLIST + SUBDAILY_DROP if c in X_q1.columns]
print(f"Columnas de satelite/sub-diarias en X del pliegue: {banned or 'ninguna'}")

# (3) Perfil por clase (diferencia estandarizada) de las continuas del contrato.
numeric_features = [
    c for c in EVERYDAY_FEATURES if pd.api.types.is_numeric_dtype(frame[c])
]
mean_pos = test_q1.loc[test_q1[LABEL_COL] == 1, numeric_features].mean()
mean_neg = test_q1.loc[test_q1[LABEL_COL] == 0, numeric_features].mean()
std_train = train_q1[numeric_features].std().replace(0, np.nan)
profile = (
    pd.DataFrame(
        {
            "media_no_fuego": mean_neg,
            "media_fuego": mean_pos,
            "dif_estandarizada": (mean_pos - mean_neg) / std_train,
        }
    )
    .sort_values("dif_estandarizada", key=lambda s: s.abs(), ascending=False)
)
display(profile.head(12))

# (4) Importancia por permutación en el pliegue.
importance = permutation_importance(
    fitted_q1,
    X_q1,
    test_q1[LABEL_COL],
    n_repeats=10,
    random_state=SEED,
    scoring="average_precision",
)
top_importance = pd.Series(
    importance.importances_mean, index=X_q1.columns
).sort_values(ascending=False)
display(top_importance.head(12).rename("caida_media_PR_AUC").to_frame())

control_pr = audit_table.set_index("held_out").loc[AUDIT_FOLD, "control_pr_auc"]
verdict = (
    "POSIBLE ARTEFACTO (el control trivial tambien separa)"
    if control_pr >= 0.9
    else "separable de verdad (el control trivial no lo reproduce)"
)
print(f"Veredicto {AUDIT_FOLD}: {verdict} | control PR-AUC={control_pr:.3f}")

,held_out,n_test,prevalence,pr_auc,roc_auc,control_pr_auc,control_roc_auc
0,2024-Q1,4158,0.267917,0.991211,0.996493,0.450508,0.709086
1,2023-Q1,8568,0.657796,0.957884,0.926785,0.855615,0.800147


Columnas de satelite/sub-diarias en X del pliegue: ninguna


,media_no_fuego,media_fuego,dif_estandarizada
relative_humidity_2m,74.256434,54.985608,-0.793031
shortwave_radiation,182.500767,404.129279,0.594872
indice_riesgo_meteo_log,2.177885,3.137684,0.578457
ratio_prevencion_extincion,0.707053,0.615776,-0.564190
temperature_2m,9.044039,13.822718,0.544419
et0_fao_evapotranspiration,0.104797,0.241533,0.518060
prevention_current,57.445861,39.926032,-0.486150
prevention_prev3,128.596373,90.640395,-0.461036
extinction_prev3,168.169159,133.092504,-0.434106
estres_termico_vegetacion,0.152289,0.339138,0.371312


,caida_media_PR_AUC
boundary_layer_height,0.333172
clc_class_6,0.023903
precipitation,0.005065
clc_class_7,0.003572
latitude,0.002793
clc_class_5,0.001472
soil_moisture_0_to_7cm,0.001226
longitude,0.001052
clc_class_9,0.000695
clc_class_3,0.000657


Veredicto 2024-Q1: separable de verdad (el control trivial no lo reproduce) | control PR-AUC=0.451


## 7 — Salidas

Predicciones out-of-fold (`record_id`, etiqueta, score, protocolo, grupo
retenido, **`risk_band`/`risk_percentile`** relativos y **`ccaa_fire_rate`**
empírica), tabla de métricas por pliegue, tabla de tasas CCAA×día; curvas PR
(OOF por protocolo), mapa de PR-AUC por CCAA, importancia por permutación del
modelo base y versiones de paquetes. El texto de métodos cierra con los límites
honestos.

_[ENG] Save out-of-fold predictions (with relative risk bands and the empirical CCAA-day rate joined), the fold-metrics table and the CCAA-day rates table; PR curves, per-CCAA choropleth, permutation importance and package versions._

In [8]:
def collect_oof(data, feature_cols, categorical_cols, model, protocol):
    """Scores out-of-fold para un protocolo (id, label, score, grupo)."""
    frames = []
    for split in blocked_splits(data, protocol):
        train, test = data.iloc[split.train_index], data.iloc[split.test_index]
        if train[LABEL_COL].nunique() < 2:
            continue
        X_train, X_test = prepare_fold(train, test, feature_cols, categorical_cols)
        fitted = clone(model).fit(X_train, train[LABEL_COL])
        out = test[["record_id", LABEL_COL]].copy()
        out["score"] = fitted.predict_proba(X_test)[:, 1]
        out["protocol"] = protocol
        out["held_out"] = split.held_out
        frames.append(out)
    return pd.concat(frames, ignore_index=True)


oof = pd.concat(
    [collect_oof(frame, EVERYDAY_FEATURES, categorical_columns(), floor_model, p) for p in PROTOCOLS],
    ignore_index=True,
)

# Tramos de riesgo (relativos, por protocolo) + tasa base CCAA×día (Plan D).
fire_rates = ccaa_day_fire_rates(positives)
oof = add_risk_bands(oof, n_bands=RISK_BANDS)
oof = join_ccaa_fire_rates(oof, frame[["record_id", "ccaa"]], fire_rates)
save_csv(oof, PREDICTIONS_CSV)
save_csv(candidate_metrics, FOLD_METRICS_CSV)
save_csv(fire_rates, CCAA_RATES_CSV)
print(f"Predicciones -> {PREDICTIONS_CSV} ({len(oof):,} filas, {oof.shape[1]} columnas)")
print(f"Metricas por pliegue -> {FOLD_METRICS_CSV}")
print(f"Tasas CCAA x dia -> {CCAA_RATES_CSV} ({len(fire_rates)} filas)")
display(oof["risk_band"].value_counts().rename_axis("risk_band").to_frame("n"))

fig, axes = plt.subplots(1, len(PROTOCOLS), figsize=(5.5 * len(PROTOCOLS), 4))
for ax, protocol in zip(np.atleast_1d(axes), PROTOCOLS):
    subset = oof[oof["protocol"] == protocol]
    PrecisionRecallDisplay.from_predictions(subset[LABEL_COL], subset["score"], ax=ax)
    ax.set_title(f"PR (OOF) — bloqueo por {protocol}")
plt.tight_layout()
plt.show()

cca_metrics = candidate_metrics[
    (candidate_metrics["protocol"] == "ccaa") & (candidate_metrics["model"] == best_name)
]
boundaries = load_boundaries()
boundaries["ccaa"] = boundaries["name"].map(REGION_NAME_MAP).fillna(boundaries["name"])
map_data = boundaries.merge(
    cca_metrics[["held_out", "pr_auc"]], left_on="ccaa", right_on="held_out", how="left"
)
fig, ax = plt.subplots(figsize=(9, 7))
map_data.plot(column="pr_auc", cmap="YlOrRd", legend=True, missing_kwds={"color": "lightgrey"}, ax=ax)
ax.set_title(f"PR-AUC por CCAA (bloqueo por ccaa, {best_name})")
ax.axis("off")
plt.tight_layout()
plt.show()

first_split = blocked_splits(frame, "season")[0]
train, test = frame.iloc[first_split.train_index], frame.iloc[first_split.test_index]
X_train, X_test = prepare_fold(train, test, EVERYDAY_FEATURES, categorical_columns())
fitted = clone(floor_model).fit(X_train, train[LABEL_COL])
importance = permutation_importance(
    fitted, X_test, test[LABEL_COL], n_repeats=10, random_state=SEED, scoring="average_precision"
)
importance_table = pd.Series(importance.importances_mean, index=X_test.columns).sort_values(ascending=False)
ax = importance_table.head(20)[::-1].plot.barh(figsize=(8, 6))
ax.set_title("Importancia por permutacion (logistic, PR-AUC)")
ax.set_xlabel("Caida media de PR-AUC")
plt.tight_layout()
plt.show()

# Tasas empíricas CCAA×día (Plan D): referencia operativa del ranking.
print("Tasas empíricas CCAA×día (Plan D):")
display(fire_rates)

print(
    f"numpy {np.__version__} | pandas {pd.__version__} | "
    f"scikit-learn {sklearn.__version__} | geopandas {geopandas.__version__} | "
    f"matplotlib {matplotlib.__version__}"
)

Predicciones -> data/processed/predictions/occurrence_scores.csv (94,074 filas, 9 columnas)
Metricas por pliegue -> data/processed/predictions/occurrence_fold_metrics.csv
Tasas CCAA x dia -> data/processed/predictions/ccaa_day_fire_rates.csv (18 filas)


,n
risk_band,
Medio,18816
Muy alto,18816
Muy bajo,18814
Bajo,18814
Alto,18814


C:\Users\Jorge\AppData\Local\Temp\ipykernel_40788\1720491019.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Jorge\AppData\Local\Temp\ipykernel_40788\1720491019.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Tasas empíricas CCAA×día (Plan D):


C:\Users\Jorge\AppData\Local\Temp\ipykernel_40788\1720491019.py:70: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,ccaa,n_ccaa_days,n_fire_days,fire_rate
0,Andalucía,731,488,0.667579
1,Castilla-La Mancha,731,363,0.496580
2,Extremadura,731,333,0.455540
3,Castilla y León,731,287,0.392613
4,Comunidad Valenciana,731,281,0.384405
5,Cataluña,731,259,0.354309
6,Aragón,731,234,0.320109
7,Región de Murcia,731,215,0.294118
8,Asturias,731,204,0.279070
9,Galicia,731,135,0.184679


numpy 2.5.2 | pandas 3.0.5 | scikit-learn 1.9.0 | geopandas 1.1.4 | matplotlib 3.11.1


## 8 — Probabilidad de excedencia calibrada (C, provisional)

Convierte el ranking en una probabilidad **honesta a escala de área**:
P(≥1 detección a ≤ R km ese día). Se entrena en 2023 y se calibra en 2024
(temporal, sin barajar); la isotónica ajusta score→probabilidad y se
**corrige el prior** por la tasa base *medida* en el pool de fondo (no el ratio
1:`background_ratio` del muestreo). La fiabilidad se calcula con
**cross-fitting** (el calibrador no se puntúa sobre su propio ajuste) y se
barren R ∈ {5, 10, 20} km. `eventos.csv` da el número de eventos únicos
(tamaño muestral honesto). Sigue siendo **provisional**: el pool de fondo no
está congelado y la distribución de evaluación es muestreada, no uniforme.

_[ENG] Calibrate the ranking into a wide-margin exceedance probability (2023 train → 2024 calibrate, isotonic + prior correction, cross-fitted reliability, margins {5,10,20} km); provisional, not a deployment probability._

In [9]:
# Ruta C (provisional): calibración de excedencia a margen ancho.
TRAIN_YEAR, CALIB_YEAR = 2023, 2024


def _year(data):
    return pd.to_datetime(data["acq_date"], errors="coerce").dt.year.to_numpy()


pos_year, bg_year = _year(positives), _year(background)
train_pos = positives.loc[pos_year == TRAIN_YEAR]
calib_pos = positives.loc[pos_year == CALIB_YEAR]
train_neg = sample_background(
    background.loc[bg_year == TRAIN_YEAR], n_positives=len(train_pos), ratio=RATIO, seed=SEED
)
calib_neg = sample_background(
    background.loc[bg_year == CALIB_YEAR], n_positives=len(calib_pos), ratio=RATIO, seed=SEED
)

train_t = assemble_modeling_frame(occurrence_features(train_pos), occurrence_features(train_neg))
calib_t = assemble_modeling_frame(occurrence_features(calib_pos), occurrence_features(calib_neg))
X_train, X_calib = prepare_fold(train_t, calib_t, EVERYDAY_FEATURES, categorical_columns())
calib_model = clone(model_candidates(seed=SEED)[best_name]).fit(X_train, train_t[LABEL_COL])

calib_t = calib_t.copy()
calib_t["acq_date"] = np.concatenate(
    [calib_pos["acq_date"].to_numpy(), calib_neg["acq_date"].to_numpy()]
)
calib_t["score"] = calib_model.predict_proba(X_calib)[:, 1]
print(
    f"Entreno {TRAIN_YEAR}: {len(train_t):,} filas | calibra {CALIB_YEAR}: "
    f"{len(calib_t):,} filas | modelo base: {best_name}"
)

results = calibrate_exceedance(
    calib_t, positives, background, margins=MARGINS, n_splits=5, seed=SEED
)
summary = exceedance_summary(results)
display(summary)

n_events = dedupe_events(positives, load_csv(EVENTOS_CSV))["event_id"].nunique()
print(f"Eventos únicos (dedup) tras los positivos: {n_events:,}")

reliability = pd.concat(
    [result.reliability.assign(margin_km=result.margin_km) for result in results.values()],
    ignore_index=True,
)
save_csv(reliability, CALIB_CSV)
print(f"Curvas de fiabilidad -> {CALIB_CSV} ({len(reliability)} filas)")

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot([0, 1], [0, 1], "k--", lw=1, label="ideal")
for margin, result in results.items():
    curve = result.reliability
    ax.plot(curve["mean_predicted"], curve["observed_freq"], "o-", label=f"R={margin:g} km")
ax.set_xlabel("probabilidad predicha (calibrada)")
ax.set_ylabel("frecuencia observada")
ax.set_title("Fiabilidad de la excedencia (isotónica, cross-fit)")
ax.legend()
plt.tight_layout()
plt.show()

C:\Projects\wildfires-project\src\wildfire\processing\occurrence.py:226: UserWarning: Ignoring 13 satellite/sub-daily columns (['acq_time', 'brightness', 'brightness_ir', 'confidence_cat', 'confidence_num', 'daynight', 'franja_horaria', 'frp', 'is_day', 'rain', 'scan', 'type']); they cannot enter the occurrence model
  warnings.warn(
C:\Projects\wildfires-project\src\wildfire\processing\occurrence.py:226: UserWarning: Ignoring 7 satellite/sub-daily columns (['acq_time', 'daynight', 'franja_horaria', 'is_day', 'rain', 'type']); they cannot enter the occurrence model
  warnings.warn(
C:\Projects\wildfires-project\src\wildfire\processing\occurrence.py:226: UserWarning: Ignoring 13 satellite/sub-daily columns (['acq_time', 'brightness', 'brightness_ir', 'confidence_cat', 'confidence_num', 'daynight', 'franja_horaria', 'frp', 'is_day', 'rain', 'scan', 'type']); they cannot enter the occurrence model
  warnings.warn(
C:\Projects\wildfires-project\src\wildfire\processing\occurrence.py:226: Us

Entreno 2023: 26,674 filas | calibra 2024: 20,363 filas | modelo base: hist_gradient_boosting


,margin_km,n_points,n_margin_positive,source_prevalence,base_rate,prior_corrected_mean,brier,log_loss,ece
0,5.0,20363,8466,0.415754,0.001297,0.002545,0.160249,0.474810,0.027649
1,10.0,20363,8511,0.417964,0.005022,0.009584,0.160633,0.476873,0.025679
2,20.0,20363,8671,0.425821,0.017912,0.032997,0.161163,0.480468,0.025541


Eventos únicos (dedup) tras los positivos: 1,178
Curvas de fiabilidad -> data/processed/predictions/calibration_reliability.csv (30 filas)


C:\Users\Jorge\AppData\Local\Temp\ipykernel_40788\1098808418.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Nota de métodos

- **Ranking, no probabilidad**: las puntuaciones ordenan; no son frecuencias
  calibradas. La prevalencia la fija el muestreo de fondo (ratio objetivo
  `background_ratio`; si el pool es menor, se usa entero), no la
  naturaleza. La PR-AUC de referencia es la prevalencia del pliegue.
- **Excedencia calibrada (sección 8, PROVISIONAL)**: el ranking se convierte en
  P(≥1 detección a ≤ R km ese día) con isotónica entrenada en 2023 y calibrada
  en 2024 (temporal), corregida por la tasa base medida en el pool de fondo; la
  fiabilidad se reporta con *cross-fitting* y márgenes R∈{5,10,20} km. No es una
  probabilidad puntual y depende del pool de fondo (no congelado).
- **Tramos de riesgo (`risk_band`)**: categorías relativas **al lote
  puntuado** (Muy bajo..Muy alto, quintiles por protocolo). No son una escala
  absoluta de peligro: un "Muy alto" es alto frente a las demás filas del
  mismo protocolo, no frente al territorio.
- **Tasa base CCAA×día (`ccaa_fire_rate`)**: P(≥1 detección | CCAA, día)
  empírica, adjunta a cada fila como referencia operativa. La calibración
  score↔probabilidad a margen ancho es **provisional** (sección 8,
  `analysis/calibration.py`); el modelo de producción se mantiene en modo
  *ranking*.
- **Auditoría 2024-Q1**: el pliegue casi perfecto se audita en la sección 6
  (control trivial de calendario/CCAA + perfil de features + permutación); el
  veredicto se imprime en la celda de auditoría.
- **Muestreo de fondo**: días de fondo por CCAA×día, con clima acotado por
  cuota de API (no todas las horas/lugares), y filtrado de fugas a
  ≤ `leakage_max_km` de una detección del mismo día.
- **Validación bloqueada**: leave-one-group-out por temporada y CCAA; el
  one-hot y el escalador se ajustan solo en entrenamiento.
- **Reintegración**: `cluster_meteo`/`score_anomalia` se ajustan solo con
  positivos; se promueven solo si la ablación lo justifica.
- **Procedencia del PCA**: verificada 2026-10-18 — `fit_weather_pca`
  reproduce el espacio de `features_a.csv`/03b (PC1–7, k=7, varianza
  acumulada 0.9152; `tests/test_pca_weather.py::TestAgainstRealFeaturesA`).
